In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)



# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_1.py ──
"""
Shared infrastructure for MLFP03 Exercise 1 — Feature Engineering
and Feature Selection on ICU data.

Contains:
    - ICU multi-table loading with temporal casts
    - Point-in-time feature builders (vitals, medications, labs)
    - ExperimentTracker / ConnectionManager setup
    - Shared prep helpers for feature-selection methods
    - Plotting helpers for feature rankings

Technique-specific code (mutual_info, RFE, Lasso paths, schema
validation) does NOT belong here — it lives in the per-technique files.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from dotenv import load_dotenv

from kailash.db import ConnectionManager
from kailash_ml import DataExplorer, ExperimentTracker



# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()

OUTPUT_DIR = Path("outputs") / "mlfp03_ex1_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_DB = "sqlite:///mlfp03_experiments.db"
EXPERIMENT_NAME = "mlfp03_healthcare_features"

_DT_FMT = "%Y-%m-%d %H:%M:%S"

# PREDICTION TIME. The model scores each admission PREDICTION_HOURS after
# ICU admission ("will this be a long stay?"). Every event-based feature
# may only use records with admit_time <= event_time <= prediction_cutoff.
# Anything later — and anything that depends on the discharge time — does
# not exist yet at prediction time.
PREDICTION_HOURS = 24

# The target is derived from los_days (known only at discharge). These
# columns are TARGET SOURCES or post-outcome information and must never be
# features. ``audit_feature_list`` enforces this.
ID_COLUMNS: frozenset[str] = frozenset(
    {"patient_id", "admission_id", "admit_time", "prediction_cutoff"}
)
TARGET_SOURCE_COLUMNS: frozenset[str] = frozenset({"los_days", "discharge_time"})
TARGET_NAME = "long_stay"

VITAL_COLS: list[str] = [
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "temperature",
    "spo2",
    "respiratory_rate",
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — ICU multi-table
# ════════════════════════════════════════════════════════════════════════


def load_icu_tables() -> dict[str, pl.DataFrame]:
    """Load all five ICU tables and cast timestamp columns to datetime.

    Returns a dict with keys: patients, admissions, vitals (long format),
    medications, labs.

    Vitals are returned in LONG format (columns: admission_id, patient_id,
    timestamp, vital_name, value) — the monolithic exercise unpivots them
    inline, but every technique file wants the long form.
    """
    loader = MLFPDataLoader()
    patients = loader.load("mlfp02", "icu_patients.parquet")
    admissions = loader.load("mlfp02", "icu_admissions.parquet")
    vitals = loader.load("mlfp02", "icu_vitals.parquet")
    medications = loader.load("mlfp02", "icu_medications.parquet")
    labs = loader.load("mlfp02", "icu_labs.parquet")

    # Cast timestamps — polars reads them as strings from the parquet
    admissions = admissions.with_columns(
        pl.col("admit_time").str.to_datetime(_DT_FMT),
        pl.col("discharge_time").str.to_datetime(_DT_FMT),
    ).with_columns(
        (pl.col("admit_time") + pl.duration(hours=PREDICTION_HOURS)).alias(
            "prediction_cutoff"
        )
    )
    medications = medications.with_columns(
        pl.col("start_time").str.to_datetime(_DT_FMT),
        pl.col("end_time").str.to_datetime(_DT_FMT),
    )
    if "timestamp" in labs.columns and labs["timestamp"].dtype == pl.String:
        labs = labs.with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    # Vitals: attach patient_id via admissions join, cast timestamp, unpivot
    vitals = vitals.join(
        admissions.select(["admission_id", "patient_id"]),
        on="admission_id",
        how="left",
    ).with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    present = [c for c in VITAL_COLS if c in vitals.columns]
    if present:
        vitals = vitals.unpivot(
            present,
            index=["admission_id", "patient_id", "timestamp"],
            variable_name="vital_name",
            value_name="value",
        )

    return {
        "patients": patients,
        "admissions": admissions,
        "vitals": vitals,
        "medications": medications,
        "labs": labs,
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE BUILDERS — point-in-time aggregates
# ════════════════════════════════════════════════════════════════════════


def events_in_prediction_window(
    events: pl.DataFrame, admissions: pl.DataFrame, time_col: str
) -> pl.DataFrame:
    """Keep only events recorded between admit_time and prediction_cutoff.

    This single filter is used by EVERY event-based feature builder, so the
    point-in-time rule lives in one place and can be audited.
    """
    return events.join(
        admissions.select("admission_id", "admit_time", "prediction_cutoff"),
        on="admission_id",
        how="inner",
    ).filter(
        (pl.col(time_col) >= pl.col("admit_time"))
        & (pl.col(time_col) <= pl.col("prediction_cutoff"))
    )


def prediction_window_report(tables: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """Audit the event data actually available at prediction time.

    For each event table: how many events / admissions fall inside the
    window, and the latest event offset (hours after admission) used.
    Computed from the same filter the builders use — not asserted.
    """
    admissions = tables["admissions"]
    rows = []
    for name, time_col in (
        ("vitals", "timestamp"),
        ("medications", "start_time"),
        ("labs", "timestamp"),
    ):
        kept = events_in_prediction_window(tables[name], admissions, time_col)
        offsets = (kept[time_col] - kept["admit_time"]).dt.total_seconds() / 3600
        rows.append(
            {
                "table": name,
                "events_total": tables[name].height,
                "events_in_window": kept.height,
                "admissions_with_events": kept["admission_id"].n_unique(),
                "admissions_total": admissions.height,
                "max_offset_hours": float(offsets.max()) if kept.height else 0.0,
            }
        )
    return pl.DataFrame(rows)


def build_vital_features(
    vitals: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Aggregate long-format vitals per admission with temporal correctness.

    Only uses vital readings recorded in the first PREDICTION_HOURS of each
    admission. Returns one row per admission with columns:
        {vital}_{mean,std,min,max,range,trend,count,cv}
    """
    filtered = events_in_prediction_window(vitals, admissions, "timestamp").sort(
        "admission_id", "timestamp"
    )

    names = sorted(filtered["vital_name"].unique().to_list())
    if not names:
        return admissions.select("admission_id")
    aggs: list[pl.DataFrame] = []
    for vital in names:
        agg = (
            filtered.filter(pl.col("vital_name") == vital)
            .group_by("admission_id")
            .agg(
                pl.col("value").mean().alias(f"{vital}_mean"),
                pl.col("value").std().alias(f"{vital}_std"),
                pl.col("value").min().alias(f"{vital}_min"),
                pl.col("value").max().alias(f"{vital}_max"),
                (pl.col("value").max() - pl.col("value").min()).alias(f"{vital}_range"),
                (pl.col("value").last() - pl.col("value").first()).alias(
                    f"{vital}_trend"
                ),
                pl.col("value").count().alias(f"{vital}_count"),
                (pl.col("value").std() / pl.col("value").mean()).alias(f"{vital}_cv"),
            )
        )
        aggs.append(agg)

    # Merge vital aggregates via full-outer join with coalesced key.
    out = aggs[0]
    for a in aggs[1:]:
        out = out.join(a, on="admission_id", how="full", coalesce=True)
    return out


def build_medication_features(
    medications: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Flag high-risk medications and count distinct drugs per admission
    (medications started within the first PREDICTION_HOURS only)."""
    return (
        events_in_prediction_window(medications, admissions, "start_time")
        .group_by("admission_id")
        .agg(
            pl.col("drug_name").n_unique().alias("n_unique_medications"),
            pl.col("drug_name").count().alias("n_medication_doses"),
            pl.col("drug_name")
            .str.contains("(?i)vasopressor|norepinephrine|dopamine")
            .any()
            .alias("received_vasopressors"),
            pl.col("drug_name")
            .str.contains("(?i)antibiotic|vancomycin|meropenem")
            .any()
            .alias("received_antibiotics"),
            pl.col("drug_name")
            .str.contains("(?i)propofol|midazolam|fentanyl")
            .any()
            .alias("received_sedation"),
        )
    )


def build_lab_features(labs: pl.DataFrame, admissions: pl.DataFrame) -> pl.DataFrame:
    """Aggregate lab results per admission with abnormal-flag counts
    (results within the first PREDICTION_HOURS only)."""
    return (
        events_in_prediction_window(labs, admissions, "timestamp")
        .group_by("admission_id")
        .agg(
            pl.col("test_name").n_unique().alias("n_unique_labs"),
            pl.col("value").count().alias("n_lab_results"),
            (pl.col("flag") != "normal").sum().alias("n_abnormal_labs"),
        )
    )


def build_full_feature_frame(tables: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """End-to-end feature matrix used by every technique file.

    Composes patients + admissions with vital / medication / lab / derived /
    interaction features. Every technique file calls this so the starting
    feature matrix is identical — only the SELECTION method differs.
    """
    patients = tables["patients"]
    admissions = tables["admissions"]

    patient_admissions = patients.join(admissions, on="patient_id", how="inner")

    features = patient_admissions.clone()
    vf = build_vital_features(tables["vitals"], admissions)
    features = features.join(vf, on="admission_id", how="left")

    # Vital stat columns (std, trend, cv, ...) are legitimately null
    # when a patient has only 1 reading for a given vital, or when a
    # vital was never sampled. Fill with 0 so downstream selection
    # methods (sklearn) see no nulls.
    vital_stat_cols = [
        c
        for c in features.columns
        if any(
            c.endswith(f"_{s}")
            for s in ("mean", "std", "min", "max", "range", "trend", "count", "cv")
        )
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0.0) for c in vital_stat_cols]
    )

    mf = build_medication_features(tables["medications"], admissions)
    features = features.join(mf, on="admission_id", how="left")

    lf = build_lab_features(tables["labs"], admissions)
    features = features.join(lf, on="admission_id", how="left")

    # Derived features. NOTE: nothing here may divide by los_days or use
    # the discharge time — the length of stay is the TARGET and is unknown
    # at prediction time. Rates use the fixed PREDICTION_HOURS window.
    features = features.with_columns(
        (pl.col("n_abnormal_labs") / pl.col("n_lab_results").clip(lower_bound=1)).alias(
            "abnormal_lab_ratio"
        ),
        (pl.col("n_medication_doses") / PREDICTION_HOURS).alias(
            "medication_doses_per_hour"
        ),
        (pl.col("n_unique_medications") > 10).alias("polypharmacy_flag"),
    )

    # Null fills for patients without meds / labs
    fill_int = [
        "n_unique_medications",
        "n_medication_doses",
        "n_unique_labs",
        "n_lab_results",
        "n_abnormal_labs",
    ]
    fill_bool = [
        "received_vasopressors",
        "received_antibiotics",
        "received_sedation",
        "polypharmacy_flag",
    ]
    fill_float = [
        "abnormal_lab_ratio",
        "medication_doses_per_hour",
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0) for c in fill_int if c in features.columns],
        *[pl.col(c).fill_null(False) for c in fill_bool if c in features.columns],
        *[pl.col(c).fill_null(0.0) for c in fill_float if c in features.columns],
    )

    # Interactions (clinical domain knowledge)
    cols = features.columns
    exprs: list[pl.Expr] = []
    if "heart_rate_mean" in cols and "systolic_bp_mean" in cols:
        exprs.append(
            (
                pl.col("heart_rate_mean")
                / pl.col("systolic_bp_mean").clip(lower_bound=1)
            ).alias("shock_index")
        )
    if "systolic_bp_mean" in cols and "diastolic_bp_mean" in cols:
        exprs.append(
            ((pl.col("systolic_bp_mean") + 2 * pl.col("diastolic_bp_mean")) / 3).alias(
                "map_mean"
            )
        )
    if "temperature_mean" in cols and "heart_rate_mean" in cols:
        exprs.append(
            (pl.col("temperature_mean") * pl.col("heart_rate_mean")).alias(
                "fever_tachycardia"
            )
        )
    exprs.append(
        (pl.col("medication_doses_per_hour") * pl.col("abnormal_lab_ratio")).alias(
            "treatment_burden_score"
        )
    )
    features = features.with_columns(*exprs)

    # Fill any nulls introduced by the interactions
    for name in (
        "shock_index",
        "map_mean",
        "fever_tachycardia",
        "treatment_burden_score",
    ):
        if name in features.columns:
            features = features.with_columns(pl.col(name).fill_null(0.0))

    return features


# ════════════════════════════════════════════════════════════════════════
# SELECTION INPUT PREP
# ════════════════════════════════════════════════════════════════════════


def audit_feature_list(feature_cols: list[str]) -> None:
    """Raise if any ID, target-source or post-outcome column is a feature.

    A real leakage gate: it FAILS (raises ValueError) instead of printing
    a reassuring message. Called by ``prepare_selection_inputs`` and by the
    validation file before anything is logged.
    """
    forbidden = [
        c
        for c in feature_cols
        if c in ID_COLUMNS
        or c in TARGET_SOURCE_COLUMNS
        or c == TARGET_NAME
        or "discharge" in c.lower()
    ]
    if forbidden:
        raise ValueError(
            f"Leakage: these columns cannot be features at prediction time: {forbidden}"
        )


def build_target(features: pl.DataFrame) -> np.ndarray:
    """``long_stay`` = 1 if the stay is longer than the median length of stay.

    Derived from los_days, which is only known at DISCHARGE — so it is the
    label, never a feature.
    """
    median_los = features["los_days"].median()
    return (features["los_days"] > median_los).cast(pl.Int64).to_numpy().ravel()


def prepare_selection_inputs(
    features: pl.DataFrame,
) -> tuple[list[str], np.ndarray, np.ndarray]:
    """Return (feature_cols, X, y) for every feature-selection method.

    - Excludes ID columns, the target and its source columns
      (``audit_feature_list`` raises if any slips through)
    - Keeps bool/int/float columns only (selection methods need numbers)
    - Replaces NaN / inf with bounded numbers
    - y is the binary ``long_stay`` label from ``build_target``
    """
    exclude = ID_COLUMNS | TARGET_SOURCE_COLUMNS | {TARGET_NAME}
    numeric_dtypes = {
        pl.Float64,
        pl.Float32,
        pl.Int64,
        pl.Int32,
        pl.UInt32,
        pl.Boolean,
    }
    feature_cols = [
        c
        for c in features.columns
        if c not in exclude and features[c].dtype in numeric_dtypes
    ]
    audit_feature_list(feature_cols)

    X = features.select(feature_cols).to_numpy().astype(np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)
    y = build_target(features)
    return feature_cols, X, y


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING
# ════════════════════════════════════════════════════════════════════════


async def setup_tracking() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Initialize ConnectionManager + ExperimentTracker (kailash-ml 1.1.1).

    Every technique file in ex_1 logs into the same experiment so selection
    runs are directly comparable. The tracker is constructed via the async
    factory; the experiment is auto-created on first ``tracker.track(...)``.
    """
    tracker = await ExperimentTracker.create(store_url=EXPERIMENT_DB)
    conn = ConnectionManager(EXPERIMENT_DB)
    await conn.initialize()
    return conn, tracker, EXPERIMENT_NAME


def setup_tracking_sync() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Sync wrapper for setup_tracking — convenience for non-async files."""
    return asyncio.run(setup_tracking())


async def log_selection_run(
    tracker: ExperimentTracker,
    experiment_id: str,
    *,
    run_name: str,
    method: str,
    selected_features: list[str],
    total_features: int,
    extra_params: dict[str, str] | None = None,
    extra_metrics: dict[str, float] | None = None,
) -> str:
    """Log a feature-selection run to ExperimentTracker. Returns run id."""
    params = {
        "method": method,
        "n_features_total": str(total_features),
        "n_features_selected": str(len(selected_features)),
        "selected_features": ",".join(selected_features[:30]),
    }
    if extra_params:
        params.update(extra_params)
    metrics = {
        "n_features_selected": float(len(selected_features)),
        "selection_ratio": float(len(selected_features)) / max(1, total_features),
    }
    if extra_metrics:
        metrics.update(extra_metrics)

    async with tracker.track(experiment=experiment_id, run_name=run_name) as run:
        await run.log_params(params)
        await run.log_metrics(metrics)
        await run.add_tag("domain", "clinical")
        await run.add_tag("selection_family", method)
        run_id = run.run_id
    return run_id


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_ranking(
    title: str, ranking: list[tuple[str, float]], *, top: int = 15
) -> None:
    """Print a ranked feature list with a simple ASCII bar chart."""
    print(f"\n=== {title} ===")
    print(f"{'Feature':<35} {'Score':>10}")
    print("-" * 48)
    if not ranking:
        print("  (empty ranking)")
        return
    # Guard against NaN/inf scores (e.g. mutual-information ties on duplicate
    # rows can yield NaN) so the ASCII bar never does int(NaN).
    finite = [abs(s) for _, s in ranking[:top] if np.isfinite(s)]
    max_score = (max(finite) if finite else 1.0) or 1.0
    for name, score in ranking[:top]:
        safe = score if np.isfinite(score) else 0.0
        bar_len = int(abs(safe) / max_score * 20)
        bar = "#" * bar_len
        print(f"  {name:<33} {safe:>10.4f}  {bar}")


def save_ranking_csv(
    ranking: list[tuple[str, float]], filename: str, score_col: str = "score"
) -> Path:
    """Persist a ranking as CSV into OUTPUT_DIR. Returns the file path."""
    path = OUTPUT_DIR / filename
    pl.DataFrame(
        {"feature": [n for n, _ in ranking], score_col: [s for _, s in ranking]}
    ).write_csv(path)
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 1.3: Wrapper Feature Selection (Recursive Feature
#                         Elimination with Random Forest)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Run Recursive Feature Elimination (RFE) around a Random Forest
#   - Understand how wrapper methods capture feature INTERACTIONS
#   - Compare RFE's selection against the filter consensus
#   - Keep RFE inside the CV folds so the elimination curve is honest
#   - Apply wrapper selection in a setting where interactions matter
#     (cardiology risk models)
#
# PREREQUISITES: 02_filter_selection.py (filter consensus built)
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Theory — why wrappers see what filters miss
#   2. Build — assemble estimator + RFE
#   3. Train — fit RFE, get ranking + support mask
#   4. Visualise — ranked table + grouped-CV elimination curve
#   5. Apply — heart-failure readmission risk at a Singapore hospital
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline



## THEORY — Wrappers Capture Interactions


In [ ]:
# A wrapper method trains a MODEL, asks the model which features
# mattered, removes the weakest, and repeats. Because the model sees all
# features together, it can learn that feature A is only useful in the
# presence of feature B — an interaction a filter method ignores.
#
# Recursive Feature Elimination (RFE) is the canonical wrapper:
#     1. Fit a Random Forest on all features.
#     2. Rank features by the forest's feature_importances_.
#     3. Drop the lowest-ranked k features.
#     4. Refit, re-rank, repeat until we hit the target feature count.
#
# The Random Forest is a strong default inside RFE because it captures
# non-linear dependencies and interactions out of the box — linear
# wrappers (LogReg RFE) miss exactly the interactions we care about.
#
# COST TRADE-OFF:
#   + captures interactions
#   + works with any model that exposes feature importances
#   - much slower than filter methods (train N models, not one score)
#   - selection is specific to the estimator — an RFE-chosen set may
#     help a Random Forest but confuse a linear model



## TASK 2 — BUILD: assemble the estimator and RFE object


In [ ]:
tables = load_icu_tables()
features = build_full_feature_frame(tables)
feature_cols, X_sel, y_binary = prepare_selection_inputs(features)

print("\n" + "=" * 70)
print("  Wrapper Selection — RFE with Random Forest")
print("=" * 70)
print(f"  Features: {len(feature_cols)}")
print(f"  Samples:  {X_sel.shape[0]}")

# TODO: Build a Random Forest estimator for RFE (100 trees, max_depth=5,
# random_state=42). Hint: RandomForestClassifier(...)
rf_estimator = ____

N_FEATURES_TO_SELECT = 15
# TODO: Wrap the estimator in RFE, keeping N_FEATURES_TO_SELECT and
# dropping 5 features per step.
rfe = ____



## TASK 3 — TRAIN: fit the RFE loop


In [ ]:
# TODO: Fit the RFE loop on the selection inputs.
____

rfe_selected = [name for name, selected in zip(feature_cols, rfe.support_) if selected]
rfe_ranking = sorted(
    [(name, int(rank)) for name, rank in zip(feature_cols, rfe.ranking_)],
    key=lambda x: x[1],
)



### Checkpoint 1


In [ ]:
assert (
    len(rfe_selected) == N_FEATURES_TO_SELECT
), f"Task 3: RFE must select {N_FEATURES_TO_SELECT}, got {len(rfe_selected)}"
assert all(
    f in feature_cols for f in rfe_selected
), "Task 3: invalid feature in RFE output"
print("\n[ok] Checkpoint 1 passed — RFE fit complete\n")



## TASK 4 — VISUALISE the ranking


In [ ]:
print("\n--- RFE Ranking (top 20; rank=1 means SELECTED) ---")
print(f"{'Feature':<35} {'Rank':>6}")
print("-" * 44)
for name, rank in rfe_ranking[:20]:
    marker = "  <-- selected" if rank == 1 else ""
    print(f"  {name:<33} {rank:>6}{marker}")

print(f"\n  Total RFE-selected features: {len(rfe_selected)}")
print(f"  Selected: {rfe_selected}")

# --- RFE elimination curve: accuracy vs number of features ---
# The curve must be HONEST: if RFE chooses features on all rows and we
# then cross-validate on those same rows, the held-out folds helped pick
# the features (selection leakage). So RFE goes INSIDE a Pipeline and is
# re-fitted within every training fold. Folds are grouped by patient_id
# — one patient can have several admissions, and the same patient must
# not sit in both train and test.
groups = features["patient_id"].to_numpy()
group_cv = GroupKFold(n_splits=3)
# TODO: accuracy of always predicting the more common class
majority_acc = ____
n_features_range = [5, 8, 10, 12, 15, 18, 20, 25]
n_features_range = [n for n in n_features_range if n <= len(feature_cols)]
elim_scores = []
for n_feat in n_features_range:
    pipe = Pipeline(
        [
            (
                "rfe",
                RFE(
                    estimator=RandomForestClassifier(
                        n_estimators=50, max_depth=5, random_state=42
                    ),
                    n_features_to_select=n_feat,
                    step=5,
                ),
            ),
            ("rf", RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)),
        ]
    )
    # TODO: grouped CV accuracy of the WHOLE pipeline (RFE refits inside
    # each fold). Hint: cross_val_score(..., cv=group_cv, groups=groups,
    # scoring="accuracy").mean()
    cv_acc = ____
    elim_scores.append(cv_acc)
    print(f"  n_features={n_feat:<3}  grouped CV accuracy={cv_acc:.4f}")
print(f"  Majority-class baseline accuracy: {majority_acc:.4f}")

fig_rfe = go.Figure()
fig_rfe.add_trace(
    go.Scatter(
        x=n_features_range,
        y=elim_scores,
        mode="lines+markers",
        marker=dict(size=10, color="#2563eb"),
        line=dict(width=3),
        name="CV Accuracy",
    )
)
best_idx = int(np.argmax(elim_scores))
fig_rfe.add_annotation(
    x=n_features_range[best_idx],
    y=elim_scores[best_idx],
    text=f"Best: {n_features_range[best_idx]} features",
    showarrow=True,
    arrowhead=2,
)
fig_rfe.update_layout(
    title="RFE Elimination Curve — Accuracy vs Number of Features",
    xaxis_title="Number of Features Selected",
    yaxis_title="3-fold grouped CV accuracy (RFE inside each fold)",
    height=450,
)
fig_rfe.add_hline(y=majority_acc, line_dash="dot", annotation_text="majority-class baseline")
rfe_path = OUTPUT_DIR / "ex1_03_rfe_elimination_curve.html"
fig_rfe.write_html(str(rfe_path))
print(f"\n  Saved: {rfe_path}")



### Checkpoint 2


In [ ]:
assert rfe_ranking[0][1] == 1, "Task 4: top-ranked features should have rank=1"
print("\n[ok] Checkpoint 2 passed — RFE ranking is well-formed\n")

# INTERPRETATION — computed: does ANY feature subset beat guessing the
# majority class?
best_acc = max(elim_scores)
print(
    f"  Best grouped CV accuracy {best_acc:.4f} vs majority baseline "
    f"{majority_acc:.4f} ({best_acc - majority_acc:+.4f})."
)
if best_acc - majority_acc < 0.01:
    print(
        "  → No subset beats the baseline: RFE still returns 15 'selected'\n"
        "    features, but on this data they are an ordering of noise. A\n"
        "    wrapper's ranking means something only when the model it wraps\n"
        "    beats a trivial baseline under honest CV."
    )
else:
    print(
        "  → Compare the selected list with the filter consensus from 02:\n"
        "    features RFE keeps but filters ranked low are candidates for\n"
        "    interaction effects."
    )



## TASK 4b — LOG the wrapper run


In [ ]:
async def log_wrapper() -> str:
    conn, tracker, exp_id = await setup_tracking()
    run_id = await log_selection_run(
        tracker,
        exp_id,
        run_name="wrapper_rfe_rf15",
        method="wrapper",
        selected_features=rfe_selected,
        total_features=len(feature_cols),
        extra_params={
            "estimator": "RandomForestClassifier",
            "n_estimators": "100",
            "max_depth": "5",
            "n_features_to_select": str(N_FEATURES_TO_SELECT),
            "step": "5",
        },
        extra_metrics={},
    )
    await conn.close()
    return run_id


run_id  = await log_wrapper()
print(f"\n  ExperimentTracker run: {run_id}")



## TASK 5 — APPLY: heart-failure readmission risk at a Singapore hospital


In [ ]:
# SCENARIO (illustrative): a Singapore cardiac centre wants a 30-day
# readmission risk model for heart-failure patients. The training set has
# ~220 candidate features across demographics, lab panels, medication
# history and procedure codes. Clinicians expect interactions to matter:
#   - ejection fraction × diuretic dose (under-diuresed weak heart)
#   - creatinine × ACE inhibitor (renal contraindication)
#   - BNP × beta-blocker dose (titration window)
# Filter methods score each factor on its own and can miss these.
#
# Why RFE + Random Forest fits:
#   - Random Forest captures interactions natively through its splits
#   - RFE iteratively removes the weakest features, re-fitting so the
#     survivors can re-combine
#   - The final 15 can be reviewed by cardiologists against guidelines
#
# ILLUSTRATIVE ARITHMETIC (assumed values): if each prevented readmission
# saves ~S$16,000 and a model cuts readmissions by 4 percentage points
# on ~3,600 heart-failure discharges a year:
#     3,600 × 0.04 × S$16,000 ≈ S$2.3M/year
# — but only if the selected features beat a trivial baseline under
# honest (in-fold, grouped) cross-validation, as checked above.
#
# LIMITATIONS:
#   - RFE is estimator-specific: the 15 features that help a Random
#     Forest may not transfer cleanly to a logistic regression
#   - Compute cost scales with dataset size; at 500K rows and 500
#     features, RFE becomes impractical and you drop to embedded
#     methods (04_embedded_selection.py)
#   - The Random Forest's feature_importances_ are biased toward
#     high-cardinality features; permutation importance is more robust
#     if cardinality varies wildly across candidates



## REFLECTION


[x] Configured a Random Forest estimator inside sklearn's RFE loop
  [x] Fit RFE and extracted the selected-feature mask + ranking
  [x] Understood how wrappers promote interaction-rich features
  [x] Logged the wrapper run to ExperimentTracker
  [x] Measured the elimination curve with RFE inside grouped CV folds

  KEY INSIGHT: Wrappers see interactions but pay a compute tax. Use them
  when you can afford the training time AND when domain knowledge tells
  you interactions matter.

  Next: 04_embedded_selection.py — Lasso regularisation, which bakes
  feature selection into the model-fitting step itself.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

